# **Visualization of CNNs: Grad-CAM**
* **Objective**: Convolutional Neural Networks are widely used on computer vision. They are powerful for processing grid-like data. However we hardly know how and why they work, due to the lack of decomposability into individually intuitive components. In this assignment, we use Grad-CAM, which highlights the regions of the input image that were important for the neural network prediction.


* NB: if `PIL` is not installed, try `conda install pillow`.
* Computations are light enough to be done on CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, datasets, transforms
import matplotlib.pyplot as plt
import pickle
import urllib.request
import cv2

import numpy as np
from PIL import Image

%matplotlib inline

## Download the Model
We provide you with a model `DenseNet-121`, already pretrained on the `ImageNet` classification dataset.
* **ImageNet**: A large dataset of photographs with 1 000 classes.
* **DenseNet-121**: A deep architecture for image classification (https://arxiv.org/abs/1608.06993)

In [ ]:
densenet121 = models.densenet121(pretrained=True)
densenet121.eval() # set the model to evaluation model
pass

In [ ]:
classes = pickle.load(urllib.request.urlopen('https://gist.githubusercontent.com/yrevar/6135f1bd8dcf2e0cc683/raw/d133d61a09d7e5a3b36b8c111a8dd5c4b5d560ee/imagenet1000_clsid_to_human.pkl'))

##classes is a dictionary with the name of each class
print(classes)

## Input Images
We provide you with 20 images from ImageNet (download link on the webpage of the course or download directly using the following command line,).<br>
In order to use the pretrained model resnet34, the input image should be normalized using `mean = [0.485, 0.456, 0.406]`, and `std = [0.229, 0.224, 0.225]`, and be resized as `(224, 224)`.

In [ ]:
def preprocess_image(dir_path):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    # Note: If the inverse normalisation is required, apply 1/x to the above object

    dataset = datasets.ImageFolder(dir_path, transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224), # resize the image to 224x224
            transforms.ToTensor(), # convert numpy.array to tensor
            normalize])) #normalize the tensor

    return (dataset)

In [ ]:
import os
if not os.path.exists("data"):
    os.mkdir("data")
if not os.path.exists("data/TP2_images"):
    os.mkdir("data/TP2_images")
    !cd data/TP2_images && wget "https://www.lri.fr/~gcharpia/deeppractice/2025/TP2/TP2_images.zip" && unzip TP2_images.zip

dir_path = "data/"
dataset = preprocess_image(dir_path)

In [ ]:
# show the orignal image
index = 5
input_image = Image.open(dataset.imgs[index][0]).convert('RGB')
plt.imshow(input_image)

In [ ]:
input_image = dataset[index][0].view(1, 3, 224, 224)
output = densenet121(input_image)
values, indices = torch.topk(output, 3)
print("Top 3-classes:", indices[0].numpy(), [classes[x] for x in indices[0].numpy()])
print("Raw class scores:", values[0].detach().numpy())

# Grad-CAM
* **Overview:** Given an image, and a category (‘tiger cat’) as input, we forward-propagate the image through the model to obtain the `raw class scores` before softmax. The gradients are set to zero for all classes except the desired class (tiger cat), which is set to 1. This signal is then backpropagated to the `rectified convolutional feature map` of interest, where we can compute the coarse Grad-CAM localization (blue heatmap).


* **To Do**: Define your own function Grad_CAM to achieve the visualization of the given images. For each image, choose the top-3 possible labels as the desired classes. Compare the heatmaps of the three classes, and conclude.

More precisely, you should provide a function: `show_grad_cam(image: torch.tensor) -> None` that displays something like this:
![output_example.png](attachment:output_example.png)
where the heatmap will be correct (here it is just an example) and the first 3 classes are the top-3 predicted classes and the last is the least probable class according to the model.

* **Comment your code**: Your code should be easy to read and follow. Please comment your code, try to use the NumPy Style Python docstrings for your functions.

* **To be submitted within 2 weeks**: this notebook, **cleaned** (i.e. without results, for file size reasons: `menu > kernel > restart and clean`), in a state ready to be executed (with or without GPU) (if one just presses 'Enter' till the end, one should obtain all the results for all images) with a few comments at the end. No additional report, just the notebook!


* **Hints**:
 + We need to record the output and grad_output of the feature maps to achieve Grad-CAM. In pytorch, the function `Hook` is defined for this purpose. Read the tutorial of [hook](https://pytorch.org/tutorials/beginner/former_torchies/nnft_tutorial.html#forward-and-backward-function-hooks) carefully.
 + More on [autograd](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) and [hooks](https://pytorch.org/tutorials/beginner/former_torchies/nnft_tutorial.html#forward-and-backward-function-hooks)
 + The pretrained model densenet doesn't have an activation function after its last layer, the output is indeed the `raw class scores`, you can use them directly.
 + Your heatmap will have the same size as the feature map. You need to scale up the heatmap to the resized image (224x224, not the original one, before the normalization) for better observation purposes. The function [`torch.nn.functional.interpolate`](https://pytorch.org/docs/stable/nn.functional.html?highlight=interpolate#torch.nn.functional.interpolate) may help.  
 + Here is the link to the paper: [Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization](https://arxiv.org/pdf/1610.02391.pdf)

Class: ‘pug, pug-dog’ | Class: ‘tabby, tabby cat’
- | -
![alt](https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/dog.jpg)| ![alt](https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/cat.jpg)

## Part 1: Grad-CAM implementation

In [ ]:
#--------------------------------------------------------------------------
                            #Find target layer
#--------------------------------------------------------------------------
#print(densenet121.forward) # to find the best layer at which to do Gradcam"
# It is denseblock4.denselayer16.conv2

#Extract target layer
target_layer = list(densenet121.features[10].children())[-1].conv2 # last layer before the fully connected layer
print(target_layer)

In [ ]:
# Your code here

def grad_cam(image, target_class_index, target_layer) -> None:
  """
  Build the heatmap of the impact of pixels in target layer on the selection of target class
  """

  # Containers for activations, gradients, and feature_map_shape
  activations = []
  gradients = []
  feature_map_shape = []

  # Hook functions
  def forward_hook_fn(module, input, output):
    activations.append(output.detach())

  def backward_hook_fn(module, grad_input, grad_output):
    gradients.append(grad_output[0].detach()) # output because we want gradients with respect to the activations

  def target_layer_output_shape_hook_fn(module, input, output):
    """
    Extract the shape of the target feature map to ultimately
    be able to scale the heatmap back to image size
    """
    feature_map_shape.append( output.shape)  # Store shape


  # Attach the hooks to the target layer
  hook_activations = target_layer.register_forward_hook(forward_hook_fn)
  hook_gradients = target_layer.register_full_backward_hook(backward_hook_fn)
  hook_feature_map_shape = target_layer.register_forward_hook(target_layer_output_shape_hook_fn)

  # Run the image through the network and save the activations of the final layer
  output = densenet121(image)

  # Define the loss with respect to the target class witout indexing
  one_hot_output = F.one_hot(torch.tensor([target_class_index]), num_classes=output.size(-1)) #create one hot array with 1 in target class
  one_hot_output = one_hot_output.to(dtype=torch.float).requires_grad_(True) #convert to float for backprop and activate gradients
  one_hot_output = torch.sum(one_hot_output * output) # extract the target output from the model

  # backpropagate the gradients
  densenet121.zero_grad()
  one_hot_output.backward()

  # Remove Hooks
  hook_activations.remove()
  hook_gradients.remove()
  hook_feature_map_shape.remove()

  #process information collected from the hooks
  activations = activations[0].numpy().squeeze(0)
  gradients = gradients[0].numpy().squeeze(0)
  feature_map_shape = feature_map_shape[0][-1] #spatial dimension d of the feature map(dxd)

  # Compute weights using gradients
  weights = np.mean(gradients, axis=(1,2))

  # Build heatmap
  heatmap = np.sum(weights[:, np.newaxis, np.newaxis] * activations, axis=0)

  # Pass heatmap through ReLU
  heatmap = np.maximum(heatmap, 0)

  # Normalize the heatmap
  heatmap /= np.max(heatmap)

  #interpolate heatmap to the correct image size
  heatmap = torch.from_numpy(heatmap.reshape(1,1,feature_map_shape,feature_map_shape)) #image is 7x7 for the last conv2 layer
  heatmap = F.interpolate(heatmap,scale_factor=224//feature_map_shape,mode='bilinear') # 224//7 = 32 for conv2 layer
  heatmap = heatmap.numpy()[0,0,:,:]

  return heatmap


In [ ]:
def merge_heatmap(img, map):
  """
  Combine  heatmap with image
  """

  # Build a map
  heatmap = cv2.applyColorMap(np.uint8(255*map), cv2.COLORMAP_JET)
  heatmap = np.float32(heatmap)/255

  # Combine  image with heatmap
  merged_image = cv2.addWeighted(heatmap, 0.5, img, 0.5, 0.0)
  merged_image = np.uint8(255*merged_image[:, :, ::-1])

  return merged_image

In [ ]:
def show_grad_cam(image, target_layer):

  def get_image_from_tensor(image):
    """
    From the normalised tensor, define an array which can be combined with the heatmap
    """
    # Squeeze extra dimensionto prepare tensor denormalisation
    img = image.squeeze(0)

    #denormalise the image
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = img*std + mean

    #put back into RGB format (H,W,3)
    img = img.permute(1, 2, 0) # permute for RGB

    #set to array
    return np.array(img) #float 32

  img = get_image_from_tensor(image)

  #rewrite the input image as the tensor input for clarity
  input = image

  #extract top 3 classes
  output = densenet121(input)
  values, indices = torch.topk(output,3)

  #plot
  f, ax = plt.subplots(1,4,figsize=(20,5))
  ax[0].imshow(img)
  ax[0].set_title('Sample '+str(i+1))
  for j in range(1,4):

    target_class_index = indices[0].numpy()[j-1]
    heatmap = grad_cam(input, target_class_index, target_layer)
    merged_img = merge_heatmap(img,heatmap)
    ax[j].imshow(merged_img)
    names = classes[target_class_index].split(',')
    ax[j].set_title(names[0])

  plt.show()

In [ ]:
for i in range(3):
  image = dataset[i][0].view(1, 3, 224, 224)
  show_grad_cam(image, target_layer = target_layer)

## Part 2: Try it on a few (1 to 3) images and comment

Gradcam shows the network is essentially looking at meaningful areas of the animals, with some interesting exceptions.

For the elephant however, it is mainly focusing on three features, hind legs, ears and back, athough it might be wrongly focusing on the enviornment when looking at the back.

When the model is wrong (porcupine predicted to be a wild boar), we see the activations are not related to the animal's body.

Interestingly, in the case of the dog, one sees the feature map essentially considers information about the feet and head of the dog.


In [ ]:
show_grad_cam(image = input_image, target_layer = target_layer)

We observe the eguyptian cat is detected when the feature map looks at the cat on the left whereas a tiger cat is detected when feature map looks at the cat on the right.

![output_example.png](attachment:output_example.png)

## Part 3: Try GradCAM on others convolutional layers, describe and comment the results

We first try with the last convolutional layer in the previous block and get nonsense.

In [ ]:
target_layer = list(densenet121.features[8].children())[-1].conv2 # last layer before the fully connected layer

for i in range(3):
  image = dataset[i][0].view(1, 3, 224, 224)
  show_grad_cam(image, target_layer = target_layer)




We then try with a  convolutional layer higher up in the last dense block:

In [ ]:
target_layer = list(densenet121.features[10].children())[-2].conv2 # last layer before the fully connected layer

for i in range(3):
  image = dataset[i][0].view(1, 3, 224, 224)
  show_grad_cam(image, target_layer = target_layer)

We see here that when looking at the impact of layer which is ot not near enough the end of the convolutional part of the networ, the features are not specialised towards classification yet as the heatmap seems to be uncorrelated to what we would cnsieder meanignful physical features.

In the case of Block 8 in Densenet, we can see the network has started to recognize some key directions, notably in the case of the second image (back of the dog).

Had we moved even further up in blocks, we would have seen what would look to the human eye like noise.

At this point it might not even make a lot of sens to average across features as they may still be looking at very different things.

In the deeper layers,all the features start to concentrate on the animal's body as the network specialises for classification.

In this specific  example, the layer even seems to be looking at more meaningful layers for the elephant. However, in the case of the porcupine, this heatmap cannot explain why the model incorreclty predicts a wild boar.

This justifies why using the layer we used was a reasonable choice.



[link text](https://)## Part 4: Try GradCAM on `9928031928.png` , describe and comment the results

In [ ]:
# Define the image path
image_path = "/content/data/TP2_images/TP2_images/9928031928.png"

# Open the image
img = Image.open(image_path).convert("RGB")  # Force 3 channels (RGB)

# Show the image
img.show()  # Opens the image in the default viewer

#Imagenet mean and std
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# Convert to tensor + normalize
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts [0, 255] → [0, 1]
    transforms.Normalize(mean=mean, std=std)  # Applies (x - mean) / std
])

# Apply transformation and reshape
img_tensor = transform(img).unsqueeze(0)

#reset the target layer
target_layer = list(densenet121.features[10].children())[-1].conv2

#apply GradCam
show_grad_cam(image = img_tensor, target_layer = target_layer)



In this case, the elephant image seems to have been noised and this seems to prevent the network from infering correctly its class as it outputs completely incorrect clases.

As one would expect this is translated in the layer activations which light up in very different regions for different classes meaning the image does not activate the learned features. It activates pixels in a way that is uncorrelated to classification.

This show how gradcam can help one visualise the robustness of a network to a perturbation.

## Part 5: What are the principal contributions of GradCAM (the answer is in the paper) ?

GradCam provides a non perturbative (no modificaion of the architecture) solution to diagnose a CNN, thus helping to build trust in models. This building of trust has been validated experimentally.

This can help distinguish between a weak and a stronger network.

Its visualisations produce reasonable results when the prediction is reasonable and unreasonable ones otherwise.

In the first case, it helps identify the meaningful image regions and in the other it helps explicit the model's biases and generalisation difficulties.





## Bonus 5: What are the main differences between DenseNet and ResNet ?

The key distinction is **concatenation** vs **addition**

Instead of adding the previous layer feature map, Densenet layers  concatenate all the previous features maps, making  for optimal parameter reuse (it thus uses fewer parameters) but requiring high memory usage.
